# 02 · Client anomaly model — train, register in Unity Catalog, batch score

**Question for support:** *which client / site / app is degraded right now, before the
client opens a ticket?*

* Features come from `gold_client_health` (5-minute windows per client × site × service):
  error rate, warn rate, and p50/p95 latency relative to the service's normal latency.
* Model: scikit-learn `IsolationForest` inside a `Pipeline` (unsupervised; no labels needed
  in production).
* Evaluation: the synthetic generator injects three known incidents
  (`src/config/incidents.json`), which act as ground truth for precision / recall.
* The model is logged to MLflow, registered as `<catalog>.<schema>.client_anomaly_model`
  with alias `champion`, then loaded back by alias to score every window into
  `gold_client_anomalies`.

In [1]:
import importlib.util
import subprocess
import sys

for pkg, mod in [("scikit-learn", "sklearn"), ("mlflow>=3", "mlflow")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [2]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    dbutils  # noqa: F821
    IN_DATABRICKS = True
except NameError:
    IN_DATABRICKS = False


def param(name: str, default: str) -> str:
    if IN_DATABRICKS:
        dbutils.widgets.text(name, default)  # noqa: F821
        return dbutils.widgets.get(name)  # noqa: F821
    return os.environ.get(name.upper(), default)


if not IN_DATABRICKS:
    from databricks.connect import DatabricksSession

    PROFILE = os.environ.get("DATABRICKS_CONFIG_PROFILE", "fevm-ai")
    spark = DatabricksSession.builder.profile(PROFILE).serverless().getOrCreate()
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", PROFILE)
    mlflow.set_tracking_uri(f"databricks://{PROFILE}")
    mlflow.set_registry_uri(f"databricks-uc://{PROFILE}")
else:
    mlflow.set_tracking_uri("databricks")
    mlflow.set_registry_uri("databricks-uc")

CATALOG = param("catalog", "accessdemo")
SCHEMA = param("schema", "dev_jesus_zaragoza_helix_support")
EXPERIMENT = param("experiment", "") or (
    f"/Users/{spark.sql('SELECT current_user()').first()[0]}/helix_client_anomaly_model")
MODEL_NAME = f"{CATALOG}.{SCHEMA}.client_anomaly_model"
FEATURES = ["error_rate", "warn_rate", "p95_ratio", "p50_ratio"]
print(f"model={MODEL_NAME} experiment={EXPERIMENT}")

model=accessdemo.dev_jesus_zaragoza_helix_support.client_anomaly_model experiment=/Users/<user>/helix_client_anomaly_model


## Load features from the gold layer

In [3]:
health = spark.sql(f"""
  WITH h AS (SELECT * FROM {CATALOG}.{SCHEMA}.gold_client_health WHERE event_count >= 10),
  base AS (
    SELECT service_name,
           percentile_approx(p95_latency_ms, 0.5) AS svc_p95_median,
           percentile_approx(p50_latency_ms, 0.5) AS svc_p50_median
    FROM h GROUP BY service_name
  )
  SELECT h.*,
         h.p95_latency_ms / b.svc_p95_median AS p95_ratio,
         h.p50_latency_ms / b.svc_p50_median AS p50_ratio
  FROM h JOIN base b USING (service_name)
""").toPandas()
health["window_start"] = pd.to_datetime(health["window_start"], utc=True)
print(f"{len(health):,} client-site-service windows; "
      f"{health.client_id.nunique()} clients, {health.site_id.nunique()} sites, "
      f"{health.service_name.nunique()} services")
print(health[FEATURES].describe().round(3).to_string())

26,000 client-site-service windows; 5 clients, 15 sites, 6 services
       error_rate  warn_rate  p95_ratio  p50_ratio
count   26000.000  26000.000  26000.000  26000.000
mean        0.016      0.050      1.026      1.012
std         0.022      0.032      0.255      0.220
min         0.000      0.000      0.540      0.678
25%         0.000      0.028      0.920      0.949
50%         0.013      0.047      1.000      1.000
75%         0.025      0.069      1.095      1.054
max         0.446      0.490      8.387      6.834


## Ground truth from the injected incidents (evaluation only — never a feature)

In [4]:
incidents = json.loads((Path.cwd().parent / "config" / "incidents.json").read_text())["incidents"]
sim_start = health["window_start"].min()
hour = (health["window_start"] - sim_start).dt.total_seconds() / 3600


def label_row(i: int) -> str | None:
    r = health.iloc[i]
    for inc in incidents:
        if (inc["start_hour"] <= hour.iloc[i] < inc["end_hour"]
                and r.client_id == inc["client_id"]
                and r.service_name in inc["services"]
                and (not inc["site_ids"] or r.site_id in inc["site_ids"])):
            return inc["incident_key"]
    return None


health["true_incident"] = [label_row(i) for i in range(len(health))]
y_true = health["true_incident"].notna().astype(int)
print(f"simulated timeline starts {sim_start}; labelled incident windows: {y_true.sum()} "
      f"of {len(health):,} ({y_true.mean():.2%})")
print(health.loc[y_true == 1].groupby("true_incident").size().to_string())

simulated timeline starts 2026-11-02 00:10:00+00:00; labelled incident windows: 69 of 26,000 (0.27%)
true_incident
INC-ARCHIVE-SLOW-101     6
INC-ESIGN-503           27
INC-UPLOAD-204          36


## Train and log to MLflow

In [5]:
mlflow.set_experiment(EXPERIMENT)
params = {"n_estimators": 300, "contamination": 0.005, "max_samples": 4096, "random_state": 7}
X = health[FEATURES].astype("float64")

with mlflow.start_run(run_name=f"isoforest_{datetime.now(timezone.utc):%Y%m%dT%H%M}") as run:
    model = Pipeline([("scale", StandardScaler()), ("iforest", IsolationForest(**params))])
    model.fit(X)
    flagged = (model.predict(X) == -1).astype(int)
    score = -model.score_samples(X)  # higher = more anomalous

    metrics = {
        "precision": precision_score(y_true, flagged),
        "recall": recall_score(y_true, flagged),
        "roc_auc": roc_auc_score(y_true, score),
        "average_precision": average_precision_score(y_true, score),
        "flagged_windows": int(flagged.sum()),
        "training_windows": int(len(X)),
    }
    mlflow.log_params(params | {"features": ",".join(FEATURES)})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(
        model,
        name="model",
        signature=infer_signature(X, model.predict(X)),
        input_example=X.head(3),
        registered_model_name=MODEL_NAME,
    )
    run_id = run.info.run_id

print(f"MLflow run {run_id}")
for k, v in metrics.items():
    print(f"  {k:18s} {v:.4f}" if isinstance(v, float) else f"  {k:18s} {v}")

recall_by_incident = (
    pd.DataFrame({"incident": health["true_incident"], "flagged": flagged})
    .dropna().groupby("incident")["flagged"].agg(["sum", "count"])
)
recall_by_incident["recall"] = (recall_by_incident["sum"] / recall_by_incident["count"]).round(3)
print("\nRecall per injected incident:")
print(recall_by_incident.to_string())

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


🔗 View Logged Model at: https://fevm-serverless-stable-2xts9s.cloud.databricks.com/ml/experiments/474499328285340/models/m-36c9cc7378fd4bd5a9ee74076b5eab1f


2026/11/02 17:19:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/11/02 17:19:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Registered model 'accessdemo.dev_jesus_zaragoza_helix_support.client_anomaly_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/7 [00:00<?, ?it/s]

🔗 Created version '4' of model 'accessdemo.dev_jesus_zaragoza_helix_support.client_anomaly_model': https://fevm-serverless-stable-2xts9s.cloud.databricks.com/explore/data/models/accessdemo/dev_jesus_zaragoza_helix_support/client_anomaly_model/version/4


🏃 View run isoforest_20261103T0119 at: https://fevm-serverless-stable-2xts9s.cloud.databricks.com/ml/experiments/474499328285340/runs/eead63019777404e803eb480989c5a9d
🧪 View experiment at: https://fevm-serverless-stable-2xts9s.cloud.databricks.com/ml/experiments/474499328285340


MLflow run eead63019777404e803eb480989c5a9d
  precision          0.5308
  recall             1.0000
  roc_auc            0.9999
  average_precision  0.9500
  flagged_windows    130
  training_windows   26000
Recall per injected incident:
                      sum  count  recall
incident                                
INC-ARCHIVE-SLOW-101    6      6     1.0
INC-ESIGN-503          27     27     1.0
INC-UPLOAD-204         36     36     1.0


## Promote to `champion` in Unity Catalog

In [6]:
client = mlflow.MlflowClient()
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'") if v.run_id == run_id)
client.set_registered_model_alias(MODEL_NAME, "champion", version)
client.update_registered_model(MODEL_NAME, description=(
    "IsolationForest over 5-minute client health windows (error rate, warn rate, latency "
    "ratios). Flags degraded client/site/service windows for Helix support triage."))
print(f"{MODEL_NAME} version {version} -> alias 'champion'")

accessdemo.dev_jesus_zaragoza_helix_support.client_anomaly_model version 4 -> alias 'champion'


## Batch score with the registered model → `gold_client_anomalies`

In [7]:
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
health["is_anomaly"] = champion.predict(X) == -1
health["anomaly_score"] = score.round(4)
health["model_version"] = int(version)
health["scored_at"] = datetime.now(timezone.utc)

out_cols = ["health_key", "window_start", "client_id", "client_name", "site_id", "service_name",
            "event_count", "error_count", "warn_count", "error_rate", "warn_rate",
            "p50_latency_ms", "p95_latency_ms", "p95_ratio", "top_error_signature",
            "anomaly_score", "is_anomaly", "model_version", "scored_at"]
anoms = health.loc[health["is_anomaly"], out_cols].sort_values("anomaly_score", ascending=False)
(spark.createDataFrame(anoms)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_client_anomalies"))
spark.sql(f"""COMMENT ON TABLE {CATALOG}.{SCHEMA}.gold_client_anomalies IS
  'Client/site/service 5-minute windows flagged by {MODEL_NAME}@champion (v{version}).'""")
print(f"wrote {len(anoms)} anomalous windows to {CATALOG}.{SCHEMA}.gold_client_anomalies")
print(anoms.head(15)[["window_start", "client_id", "site_id", "service_name", "error_rate",
                      "p95_latency_ms", "top_error_signature", "anomaly_score"]].to_string(index=False))

wrote 130 anomalous windows to accessdemo.dev_jesus_zaragoza_helix_support.gold_client_anomalies
             window_start     client_id  site_id     service_name  error_rate  p95_latency_ms      top_error_signature  anomaly_score
2026-11-02 03:30:00+00:00 northwind-bio site-101  imaging-archive      0.0208          1935.1              S3 put slow         0.8465
2026-11-02 03:25:00+00:00 northwind-bio site-101  imaging-archive      0.0465          1800.7              S3 put slow         0.8435
2026-11-02 03:15:00+00:00 northwind-bio site-101  imaging-archive      0.0435          2272.5              S3 put slow         0.8420
2026-11-02 03:20:00+00:00 northwind-bio site-101  imaging-archive      0.0366          1674.0              S3 put slow         0.8409
2026-11-02 03:35:00+00:00 northwind-bio site-101  imaging-archive      0.0444          1786.0              S3 put slow         0.8376
2026-11-02 18:40:00+00:00  fabrikam-cro site-204     dicom-router      0.2203          1386.2      

In [8]:
result = {"run_id": run_id, "model": MODEL_NAME, "version": int(version),
          "anomalies": int(len(anoms)), **{k: round(float(v), 4) for k, v in metrics.items()}}
print(json.dumps(result, indent=2))
if IN_DATABRICKS:
    dbutils.notebook.exit(json.dumps(result))  # noqa: F821

{
  "run_id": "eead63019777404e803eb480989c5a9d",
  "model": "accessdemo.dev_jesus_zaragoza_helix_support.client_anomaly_model",
  "version": 4,
  "anomalies": 130,
  "precision": 0.5308,
  "recall": 1.0,
  "roc_auc": 0.9999,
  "average_precision": 0.95,
  "flagged_windows": 130.0,
  "training_windows": 26000.0
}
